In [4]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [5]:
# 데이터 로드
df = pd.read_csv('extracted_data.csv')

# ----------------------------------
# 1. 마지막 조사 데이터 제거
# ----------------------------------

df = df[df['churn'].notna()]

In [6]:
# ----------------------------------
# 파생변수 생성
# ----------------------------------

df['installment_ratio'] = (
    df['monthly_installment'] /
    (df['monthly_total_cost'] + 1)
)  # 할부금 비율

df['cost_income_ratio'] = (
    df['monthly_total_cost'] /
    (df['income'] + 1)
)  # 통신비 대비 소득

df['income_per_person'] = (
    df['income'] /
    df['household_size']
)  # 1인당 소득

df['cost_per_person'] = (
    df['monthly_total_cost'] /
    df['household_size']
)  # 1인당 통신비

df['married_large_family'] = (
    (df['marriage'] == 2) &
    (df['household_size'] >= 3)
).astype(int)  # 결혼 + 가구원수

In [7]:
# ----------------------------------
# 2. 변수 구분
# ----------------------------------

numeric_features = [
    'year',
    'age',
    'income',
    'monthly_total_cost',
    'monthly_installment',
    'household_size',
    'installment_ratio',
    'cost_income_ratio',
    'income_per_person',
    'cost_per_person'
]

categorical_features = [
    'gender',
    'school',
    'area',
    'job',
    'marriage',
    'cost_payer',
    'provider',
    'married_large_family',
    'is_mobile_bundled'
]

FEATURE_COLS = numeric_features + categorical_features

# ----------------------------------
# 3. 수치형 파이프라인
# ----------------------------------

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]
)

# ----------------------------------
# 4. 범주형 파이프라인
# ----------------------------------

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

# ----------------------------------
# 5. 전체 전처리
# ----------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

In [8]:
# --------------------------
# 6. 연도 기준 분할
# --------------------------

train_df = df[df['year'] <= 20]

valid_df = df[
    (df['year'] >= 21) &
    (df['year'] <= 22)
]

test_df = df[df['year'] >= 23]

print(len(train_df))
print(len(valid_df))
print(len(test_df))

91378
18823
8135


In [9]:
# --------------------------
# 7. 전처리 적용
# --------------------------

X_train = train_df.drop(columns=['id', 'churn'])
y_train = train_df['churn']

X_valid = valid_df.drop(columns=['id', 'churn'])
y_valid = valid_df['churn']

X_test = test_df.drop(columns=['id', 'churn'])
y_test = test_df['churn']

X_train = preprocessor.fit_transform(X_train)
X_valid = preprocessor.transform(X_valid)
X_test  = preprocessor.transform(X_test)

In [10]:
# --------------------------
# 평가 함수 정의
# --------------------------

def evaluate_model(model, X, y, dataset_name):
    y_pred = model.predict(X)

    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X)[:, 1]
    else:
        y_prob = model.predict(X)

    print(f'\n[{dataset_name}]')
    print('Accuracy :', round(accuracy_score(y, y_pred), 4))
    print('Precision:', round(precision_score(y, y_pred, zero_division=0), 4))
    print('Recall   :', round(recall_score(y, y_pred, zero_division=0), 4))
    print('F1-score :', round(f1_score(y, y_pred, zero_division=0), 4))
    print('ROC-AUC  :', round(roc_auc_score(y, y_prob), 4))


def get_feature_importance(model, preprocessor, top_n=20):
    feature_names = preprocessor.get_feature_names_out()

    importance_df = pd.DataFrame({
        'feature'   : feature_names,
        'importance': model.feature_importances_
    })

    importance_df = importance_df.sort_values(
        by='importance',
        ascending=False
    ).reset_index(drop=True)

    print(importance_df.head(top_n))
    return importance_df

In [11]:
# =====================================================
# 하이퍼파라미터 설정 (실험할 때 여기만 수정하세요)
# =====================================================

# --------------------------
# Logistic Regression
# C: 규제 강도 역수. 작을수록 규제 강함
#    후보: 0.01 / 0.1 / 1.0 / 10.0
# --------------------------
LR_C        = 1.0
LR_MAX_ITER = 1000

# --------------------------
# Random Forest
# n_estimators    : 트리 수              후보: 100 / 300 / 500
# max_depth       : None = 제한없음      후보: 5 / 8 / 10 / 15 / None
# min_samples_leaf: 클수록 과적합 방지   후보: 1 / 3 / 5 / 10
# --------------------------
RF_N_ESTIMATORS     = 300
RF_MAX_DEPTH        = 10
RF_MIN_SAMPLES_LEAF = 5

# --------------------------
# Gradient Boosting
# subsample < 1.0 이면 Stochastic GBM (과적합 방지)
# learning_rate 낮추면 n_estimators도 늘려야 함
# --------------------------
GB_N_ESTIMATORS  = 300
GB_MAX_DEPTH     = 5
GB_LEARNING_RATE = 0.05
GB_SUBSAMPLE     = 0.8

# --------------------------
# LightGBM
# num_leaves: max_depth보다 중요   후보: 15 / 31 / 63 / 127
# ※ num_leaves > 2^max_depth 이면 과적합
# --------------------------
LGB_N_ESTIMATORS      = 1000
LGB_LEARNING_RATE     = 0.02
LGB_NUM_LEAVES        = 63
LGB_MIN_CHILD_SAMPLES = 100
LGB_SUBSAMPLE         = 0.8
LGB_COLSAMPLE         = 0.8
LGB_REG_ALPHA         = 1.0
LGB_REG_LAMBDA        = 1.0

# --------------------------
# XGBoost
# scale_pos_weight: 비이탈 수 / 이탈 수   후보: 1 / 3 / 5 / 7
# min_child_weight: 클수록 과적합 방지    후보: 1 / 5 / 10 / 20
# gamma           : 분기 최소 손실 감소   후보: 0 / 0.5 / 1 / 5
# --------------------------
XGB_N_ESTIMATORS     = 1000
XGB_LEARNING_RATE    = 0.03
XGB_MAX_DEPTH        = 4
XGB_MIN_CHILD_WEIGHT = 10
XGB_SUBSAMPLE        = 0.8
XGB_COLSAMPLE        = 0.8
XGB_GAMMA            = 1
XGB_REG_ALPHA        = 1
XGB_REG_LAMBDA       = 2

---
## Logistic Regression

In [12]:
# --------------------------
# Logistic Regression 학습
# --------------------------

lr_model = LogisticRegression(
    C=LR_C,
    max_iter=LR_MAX_ITER,
    class_weight='balanced',
    random_state=42
)

lr_model.fit(X_train, y_train)

print('train finish')

train finish


In [13]:
# --------------------------
# Logistic Regression 평가
# --------------------------

evaluate_model(lr_model, X_train, y_train, 'Train')
evaluate_model(lr_model, X_valid, y_valid, 'Validation')
evaluate_model(lr_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.5883
Precision: 0.4583
Recall   : 0.5916
F1-score : 0.5165
ROC-AUC  : 0.6295

[Validation]
Accuracy : 0.5231
Precision: 0.4123
Recall   : 0.7468
F1-score : 0.5313
ROC-AUC  : 0.6235

[Test]
Accuracy : 0.5216
Precision: 0.4295
Recall   : 0.7436
F1-score : 0.5445
ROC-AUC  : 0.6084


---
## Random Forest

In [14]:
# --------------------------
# Random Forest 학습
# --------------------------

rf_model = RandomForestClassifier(
    n_estimators=RF_N_ESTIMATORS,
    max_depth=RF_MAX_DEPTH,
    min_samples_leaf=RF_MIN_SAMPLES_LEAF,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

rf_model.fit(X_train, y_train)

print('train finish')

train finish


In [15]:
# --------------------------
# Random Forest 평가
# --------------------------

evaluate_model(rf_model, X_train, y_train, 'Train')
evaluate_model(rf_model, X_valid, y_valid, 'Validation')
evaluate_model(rf_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.6419
Precision: 0.5164
Recall   : 0.5763
F1-score : 0.5447
ROC-AUC  : 0.6867

[Validation]
Accuracy : 0.5933
Precision: 0.4542
Recall   : 0.6133
F1-score : 0.5219
ROC-AUC  : 0.6404

[Test]
Accuracy : 0.5677
Precision: 0.4503
Recall   : 0.5636
F1-score : 0.5006
ROC-AUC  : 0.6127


In [16]:
# --------------------------
# Random Forest 중요 변수 확인
# --------------------------

rf_importance_df = get_feature_importance(rf_model, preprocessor, top_n=20)

                       feature  importance
0            cat__provider_1.0    0.201892
1                    num__year    0.127792
2            cat__provider_3.0    0.074287
3            cat__provider_2.0    0.052593
4      num__monthly_total_cost    0.038830
5       num__cost_income_ratio    0.037864
6         num__cost_per_person    0.037255
7       num__installment_ratio    0.036724
8               cat__area_12.0    0.035498
9     num__monthly_installment    0.031983
10                    num__age    0.026832
11               cat__area_3.0    0.023150
12               cat__area_1.0    0.021788
13              cat__area_13.0    0.021600
14               cat__area_5.0    0.018807
15      num__income_per_person    0.017259
16                 num__income    0.014958
17  cat__is_mobile_bundled_1.0    0.013720
18  cat__is_mobile_bundled_2.0    0.011632
19              cat__area_14.0    0.010539


---
## Gradient Boosting

In [17]:
# --------------------------
# Gradient Boosting 학습
# --------------------------

gb_model = GradientBoostingClassifier(
    n_estimators=GB_N_ESTIMATORS,
    max_depth=GB_MAX_DEPTH,
    learning_rate=GB_LEARNING_RATE,
    subsample=GB_SUBSAMPLE,
    random_state=42
)

gb_model.fit(X_train, y_train)

print('train finish')

train finish


In [18]:
# --------------------------
# Gradient Boosting 평가
# --------------------------

evaluate_model(gb_model, X_train, y_train, 'Train')
evaluate_model(gb_model, X_valid, y_valid, 'Validation')
evaluate_model(gb_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.6757
Precision: 0.6615
Recall   : 0.2614
F1-score : 0.3747
ROC-AUC  : 0.7036

[Validation]
Accuracy : 0.6582
Precision: 0.5613
Recall   : 0.254
F1-score : 0.3497
ROC-AUC  : 0.6498

[Test]
Accuracy : 0.6395
Precision: 0.5786
Recall   : 0.2295
F1-score : 0.3287
ROC-AUC  : 0.6323


In [19]:
# --------------------------
# Gradient Boosting 중요 변수 확인
# --------------------------

gb_importance_df = get_feature_importance(gb_model, preprocessor, top_n=20)

                     feature  importance
0          cat__provider_1.0    0.199619
1                  num__year    0.146926
2       num__cost_per_person    0.049077
3     num__cost_income_ratio    0.046883
4     num__installment_ratio    0.046185
5             cat__area_12.0    0.043210
6    num__monthly_total_cost    0.040091
7                   num__age    0.039080
8   num__monthly_installment    0.035874
9              cat__area_1.0    0.035547
10            cat__area_13.0    0.034092
11             cat__area_5.0    0.032068
12             cat__area_3.0    0.019242
13    num__income_per_person    0.014159
14            cat__area_14.0    0.011568
15             cat__area_9.0    0.011353
16         cat__provider_2.0    0.011246
17         cat__provider_3.0    0.011176
18               num__income    0.011028
19             cat__area_4.0    0.010354


---
## LightGBM

In [20]:
# --------------------------
# LightGBM 학습
# --------------------------

lgb_model = LGBMClassifier(
    objective='binary',
    n_estimators=LGB_N_ESTIMATORS,
    learning_rate=LGB_LEARNING_RATE,
    max_depth=-1,
    num_leaves=LGB_NUM_LEAVES,
    min_child_samples=LGB_MIN_CHILD_SAMPLES,
    subsample=LGB_SUBSAMPLE,
    colsample_bytree=LGB_COLSAMPLE,
    reg_alpha=LGB_REG_ALPHA,
    reg_lambda=LGB_REG_LAMBDA,
    class_weight='balanced',
    random_state=42,
    verbose=-1
)

lgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)]
)

print('train finish')

train finish


In [21]:
# --------------------------
# LightGBM 평가
# --------------------------

evaluate_model(lgb_model, X_train, y_train, 'Train')
evaluate_model(lgb_model, X_valid, y_valid, 'Validation')
evaluate_model(lgb_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.6916
Precision: 0.5724
Recall   : 0.673
F1-score : 0.6187
ROC-AUC  : 0.7523

[Validation]
Accuracy : 0.6041
Precision: 0.4635
Recall   : 0.5957
F1-score : 0.5213
ROC-AUC  : 0.6481

[Test]
Accuracy : 0.5889
Precision: 0.4705
Recall   : 0.5499
F1-score : 0.5071
ROC-AUC  : 0.6278


In [22]:
# --------------------------
# LightGBM 중요 변수 확인
# --------------------------

lgb_importance_df = get_feature_importance(lgb_model, preprocessor, top_n=20)

                       feature  importance
0       num__cost_income_ratio        8268
1         num__cost_per_person        7124
2      num__monthly_total_cost        6761
3                    num__year        6621
4       num__installment_ratio        6530
5     num__monthly_installment        4058
6                     num__age        3731
7       num__income_per_person        2242
8                  num__income        1585
9            cat__provider_1.0         736
10  cat__is_mobile_bundled_1.0         662
11               cat__area_1.0         608
12               cat__area_8.0         597
13             cat__school_5.0         591
14           cat__provider_2.0         588
15           cat__provider_3.0         578
16             cat__school_4.0         568
17           cat__marriage_2.0         500
18         num__household_size         487
19             cat__school_2.0         480


---
## XGBoost

In [23]:
# --------------------------
# XGBoost 학습
# --------------------------

xgb_model = XGBClassifier(
    n_estimators=XGB_N_ESTIMATORS,
    learning_rate=XGB_LEARNING_RATE,
    max_depth=XGB_MAX_DEPTH,
    min_child_weight=XGB_MIN_CHILD_WEIGHT,
    subsample=XGB_SUBSAMPLE,
    colsample_bytree=XGB_COLSAMPLE,
    objective='binary:logistic',
    gamma=XGB_GAMMA,
    reg_alpha=XGB_REG_ALPHA,
    reg_lambda=XGB_REG_LAMBDA,
    eval_metric='auc',
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

print('train finish')

train finish


In [24]:
# --------------------------
# XGBoost 평가
# --------------------------

evaluate_model(xgb_model, X_train, y_train, 'Train')
evaluate_model(xgb_model, X_valid, y_valid, 'Validation')
evaluate_model(xgb_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.672
Precision: 0.6441
Recall   : 0.263
F1-score : 0.3735
ROC-AUC  : 0.692

[Validation]
Accuracy : 0.6596
Precision: 0.5623
Recall   : 0.2682
F1-score : 0.3632
ROC-AUC  : 0.6501

[Test]
Accuracy : 0.6377
Precision: 0.5678
Recall   : 0.2423
F1-score : 0.3397
ROC-AUC  : 0.6339


In [25]:
# --------------------------
# XGBoost 중요 변수 확인
# --------------------------

xgb_importance_df = get_feature_importance(xgb_model, preprocessor, top_n=20)

                       feature  importance
0            cat__provider_1.0    0.155034
1               cat__area_13.0    0.065009
2               cat__area_12.0    0.064599
3                cat__area_5.0    0.062856
4                cat__area_1.0    0.044535
5                cat__area_3.0    0.035425
6            cat__provider_5.0    0.032761
7                    num__year    0.025931
8               cat__area_14.0    0.021764
9                cat__area_9.0    0.019713
10              cat__area_15.0    0.019030
11           cat__provider_3.0    0.018893
12           cat__provider_4.0    0.018539
13              cat__area_16.0    0.017994
14               cat__area_4.0    0.016555
15               cat__area_7.0    0.016178
16              cat__area_10.0    0.015347
17                    num__age    0.015324
18  cat__is_mobile_bundled_1.0    0.013842
19           cat__provider_2.0    0.013740


---
## Voting Ensemble

In [26]:
# --------------------------
# Voting Ensemble 학습
# --------------------------
# 여러 모델의 확률 예측을 평균내는 소프트 보팅
# 개별 모델보다 안정적인 성능이 나오는 경우가 많음

voting_model = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(
            n_estimators=RF_N_ESTIMATORS,
            max_depth=RF_MAX_DEPTH,
            class_weight='balanced',
            n_jobs=-1,
            random_state=42
        )),
        ('lgb', LGBMClassifier(
            n_estimators=LGB_N_ESTIMATORS,
            learning_rate=LGB_LEARNING_RATE,
            num_leaves=LGB_NUM_LEAVES,
            class_weight='balanced',
            random_state=42,
            verbose=-1
        )),
        ('xgb', XGBClassifier(
            n_estimators=XGB_N_ESTIMATORS,
            learning_rate=XGB_LEARNING_RATE,
            max_depth=XGB_MAX_DEPTH,
            eval_metric='auc',
            random_state=42,
            verbosity=0
        ))
    ],
    voting='soft',
    n_jobs=-1
)

voting_model.fit(X_train, y_train)

print('train finish')

train finish


In [27]:
# --------------------------
# Voting Ensemble 평가
# --------------------------

evaluate_model(voting_model, X_train, y_train, 'Train')
evaluate_model(voting_model, X_valid, y_valid, 'Validation')
evaluate_model(voting_model, X_test,  y_test,  'Test')


[Train]
Accuracy : 0.6916
Precision: 0.6042
Recall   : 0.4939
F1-score : 0.5435
ROC-AUC  : 0.7325

[Validation]
Accuracy : 0.6419
Precision: 0.5056
Recall   : 0.4731
F1-score : 0.4888
ROC-AUC  : 0.6543

[Test]
Accuracy : 0.6135
Precision: 0.497
Recall   : 0.4277
F1-score : 0.4598
ROC-AUC  : 0.6312


In [28]:
import joblib
import os
from pathlib import Path

# 저장할 models 폴더가 없으면 자동으로 생성
save_dir = '../models'

Path(save_dir).mkdir( exist_ok=True)
print(f"'{save_dir}' 폴더가 새로 생성되었습니다.")

# 학습 완료된 모델(또는 파이프라인) 객체들을 딕셔너리로 묶는다.
trained_models = {
    "lr": lr_model,
    "rf": rf_model,
    "gb": gb_model,
    "lgb": lgb_model,
    "xgb": xgb_model,
    "voting": voting_model
}

print("\n 모델별 joblib 파일 저장 시작..")

# 반복문을 돌며 개별 파일로 저장
for prefix, model_obj in trained_models.items():
    # 파일명 정의 (예: models/lr_churn_model.joblib)
    file_name = f"{prefix}_churn_model.joblib"
    file_path = os.path.join(save_dir, file_name)

    joblib.dump(model_obj, file_path)
    print(f"[{prefix.upper()}] 모델 저장 완료 -> {file_path}")

print("\n 모든 모델이 models 폴더에 저장 완료")

'../models' 폴더가 새로 생성되었습니다.

 모델별 joblib 파일 저장 시작..
[LR] 모델 저장 완료 -> ../models\lr_churn_model.joblib
[RF] 모델 저장 완료 -> ../models\rf_churn_model.joblib
[GB] 모델 저장 완료 -> ../models\gb_churn_model.joblib
[LGB] 모델 저장 완료 -> ../models\lgb_churn_model.joblib
[XGB] 모델 저장 완료 -> ../models\xgb_churn_model.joblib
[VOTING] 모델 저장 완료 -> ../models\voting_churn_model.joblib

 모든 모델이 models 폴더에 저장 완료
